# Play PRISM on rough terrain

Run the cells from top to bottom to compare pretrained **PRISM** and **MLP** G1 policies.
Change friction, robot mass, payload or velocity and watch the resulting native simulation.

**A GPU runtime is recommended** (Runtime → Change runtime type → T4 GPU).
A CPU runtime also works, with lower video frame rate and longer execution time.
No training or dataset is needed. The first installation
and MuJoCo-Warp compilation take a few minutes. This is a qualitative playground;
the paper's aggregate evaluation uses a separate protocol.

[Project](https://lsh3163.github.io/prism/) · [Code and weights](https://github.com/lsh3163/prism)


## 1. Install

The example uses its own Python environment, leaving Colab's notebook packages intact.


In [ ]:
import os
import shutil
import subprocess
import sys
from pathlib import Path

from IPython.display import HTML, display

subprocess.run([sys.executable, "-m", "pip", "install", "-q", "uv==0.11.7"], check=True)
uv = shutil.which("uv")
assert uv, "uv was not installed"
subprocess.run(["apt-get", "-qq", "update"], check=True)
subprocess.run(["apt-get", "-qq", "install", "-y", "libosmesa6"], check=True)
repo = Path("/content/prism")
if not repo.exists():
    subprocess.run(
        ["git", "clone", "--depth", "1", "https://github.com/lsh3163/prism.git", str(repo)], check=True
    )
if not (repo / ".venv/bin/python").exists():
    subprocess.run([uv, "venv", "--python", "3.13", str(repo / ".venv")], check=True)
python = str(repo / ".venv/bin/python")
try:
    has_gpu = subprocess.run(["nvidia-smi", "-L"], capture_output=True).returncode == 0
except FileNotFoundError:
    has_gpu = False
install_command = [
    uv,
    "pip",
    "install",
    "--python",
    python,
    "-e",
    str(repo),
    "-r",
    str(repo / "examples/g1/requirements.txt"),
]
try:
    subprocess.run([*install_command, "--torch-backend", "auto" if has_gpu else "cpu"], check=True)
except subprocess.CalledProcessError:
    if not has_gpu:
        raise
    print("GPU package resolution failed; installing the CPU runtime.")
    subprocess.run([*install_command, "--torch-backend", "cpu"], check=True)
probe = subprocess.run(
    [python, "-c", "import torch; print('cuda:0' if torch.cuda.is_available() else 'cpu')"],
    check=True,
    capture_output=True,
    text=True,
)
device = probe.stdout.strip().splitlines()[-1]
render_backend = "egl" if device.startswith("cuda") else "osmesa"
run_env = dict(
    os.environ,
    MUJOCO_GL=render_backend,
    PYOPENGL_PLATFORM=render_backend,
    MJLAB_WARP_QUIET="1",
    PYTHONUNBUFFERED="1",
    LP_NUM_THREADS="1",
    OMP_NUM_THREADS="1",
    OPENBLAS_NUM_THREADS="1",
    MKL_NUM_THREADS="1",
)
print(f"Ready on {device}: pretrained PRISM and MLP weights are included.")
print("CPU videos use 5 fps; CUDA videos use 25 fps.")

## 2. Choose a condition

Defaults compare both actors on low-friction rough terrain. Both receive the same
terrain seed and reset. A positive payload appears as a backpack. A trial ends when
the robot falls or reaches the time limit.


In [ ]:
terrain = "rough"  # @param ["rough", "flat"]
friction = 0.2  # @param {type:"number"}
mass_ratio = 1.0  # @param {type:"number"}
payload_kg = 0.0  # @param {type:"number"}
forward_velocity = 0.5  # @param {type:"number"}
sideways_velocity = 0.0  # @param {type:"number"}
yaw_velocity = 0.0  # @param {type:"number"}
seconds = 20.0  # @param {type:"number"}
seed = 401  # @param {type:"integer"}

options = [
    "--terrain",
    terrain,
    "--friction",
    str(friction),
    "--mass-ratio",
    str(mass_ratio),
    "--payload",
    str(payload_kg),
    "--velocity",
    str(forward_velocity),
    str(sideways_velocity),
    str(yaw_velocity),
    "--seed",
    str(seed),
    "--device",
    device,
]

## 3. Run PRISM and MLP

This generates fresh rollouts from the released policies. Kernel compilation happens on
the first run. Progress is saved in the log files; a fall stops that actor's video.


In [ ]:
import json
import time

videos = {}
results = {}
for actor in ("mlp", "prism"):
    movie = repo / "outputs/g1" / f"{actor}.mp4"
    log = movie.with_suffix(".log")
    movie.parent.mkdir(parents=True, exist_ok=True)
    print(f"Running {actor.upper()}...", flush=True)
    started = time.monotonic()
    with log.open("w") as stream:
        completed = subprocess.run(
            [
                python,
                "-u",
                str(repo / "examples/g1/play.py"),
                "--actor",
                actor,
                *options,
                "--headless",
                "--seconds",
                str(seconds),
                "--video",
                str(movie),
            ],
            cwd=repo,
            env=run_env,
            stdout=stream,
            stderr=subprocess.STDOUT,
        )
    if completed.returncode:
        print(log.read_text()[-6000:])
        raise RuntimeError(f"{actor} failed; see {log}")
    rows = [json.loads(line) for line in log.read_text().splitlines() if line.startswith('{"actor":')]
    if not rows or not movie.is_file():
        raise RuntimeError(f"Missing rollout result or movie: {log}")
    results[actor] = rows[-1]
    videos[actor] = movie
    result = results[actor]
    status = "fell" if result["fell_over"] else "reached the time limit"
    print(
        f"{actor.upper()}: {status} at {result['elapsed_seconds']:.2f} s; "
        f"forward distance {result['forward_distance_m']:.2f} m "
        f"(wall time {time.monotonic() - started:.0f} s)",
        flush=True,
    )
assert results["mlp"]["initial_state_sha256"] == results["prism"]["initial_state_sha256"], (
    "Initial states differ"
)
assert results["mlp"]["terrain_sha256"] == results["prism"]["terrain_sha256"], "Terrain geometry differs"
print("Verified: both actors used the same initial state and terrain.")

## 4. Watch the comparison

Videos have independent controls because a fallen policy stops earlier. Change the
condition in step 2 and rerun steps 3–4 for another comparison.


In [ ]:
import base64
import html

panels = []
for actor in ("mlp", "prism"):
    encoded = base64.b64encode(videos[actor].read_bytes()).decode()
    result = results[actor]
    status = "Fell" if result["fell_over"] else "Reached horizon"
    caption = f"{status} · {result['elapsed_seconds']:.2f} s · {result['forward_distance_m']:.2f} m forward"
    panels.append(
        '<div style="flex:1;min-width:280px"><h3>' + actor.upper() + "</h3>"
        '<video controls playsinline style="width:100%;border-radius:8px" '
        'src="data:video/mp4;base64,' + encoded + '"></video><p>' + html.escape(caption) + "</p></div>"
    )
display(HTML('<div style="display:flex;gap:16px;flex-wrap:wrap">' + "".join(panels) + "</div>"))

## 5. Optional: interactive 3D viewer

Run this cell to pause, reset, rotate the camera and change the command in a live viewer.
This uses Colab's own [kernel-port iframe](https://github.com/googlecolab/colabtools/blob/main/google/colab/output/_util.py).
Run the last cell to stop it before generating more videos.


In [ ]:
import urllib.request

from google.colab import output

actor = "prism"  # @param ["prism", "mlp"]

previous_viewer = globals().get("viewer_process")
if previous_viewer is not None and previous_viewer.poll() is None:
    previous_viewer.terminate()
    previous_viewer.wait(timeout=20)
viewer_log = repo / "outputs/g1/viewer.log"
viewer_log.parent.mkdir(parents=True, exist_ok=True)
with viewer_log.open("w") as stream:
    viewer_process = subprocess.Popen(
        [
            python,
            "-u",
            str(repo / "examples/g1/play.py"),
            "--actor",
            actor,
            *options,
            "--port",
            "8080",
        ],
        cwd=repo,
        env=run_env,
        stdout=stream,
        stderr=subprocess.STDOUT,
    )

for _ in range(240):
    if viewer_process.poll() is not None:
        raise RuntimeError(viewer_log.read_text()[-6000:])
    try:
        with urllib.request.urlopen("http://127.0.0.1:8080", timeout=1) as response:
            if response.status == 200:
                break
    except OSError:
        time.sleep(1)
else:
    viewer_process.terminate()
    raise TimeoutError(f"Viewer not ready; see {viewer_log}")

output.serve_kernel_port_as_iframe(8080, height=650)

In [ ]:
previous_viewer = globals().get("viewer_process")
if previous_viewer is not None and previous_viewer.poll() is None:
    previous_viewer.terminate()
    previous_viewer.wait(timeout=20)
print("Viewer stopped.")